# v9_b Step 5d: All possible reaction paths to final annotated M0,[M+H]+ features

This is a **read-only post-processing analysis** of the existing v9_b results. The two Step 5b reaction annotation tables define the endpoint features; four Step 3 JSON files retain each fraction's original ID, m/z and candidate substrate; five Step 4 JSON files contain the predicted 1–3 step reaction chains. No Khipu or network step is rerun.

**Counting rules**

- Match each eligible endpoint side to every product from its experimental steroid parent at **strictly <10 ppm**, using the feature m/z as denominator. Predicted RT is unknown, so RT is not a path filter.
- Keep 1-, 2- and 3-step paths simultaneously. A path is the **ordered sequence of elemental formula differences**. Equivalent spelling, such as `H2` and `+H2`, is deduplicated; reaction names are reported separately and are not multiplied into paths. A path cannot revisit a molecular formula already on its chain (for example `+O → −O → +O`), because the formula-level network cannot distinguish a return to the parent from a new state.
- A merged pellets/supernatant endpoint counts once per parent/path; both measured sides remain traceable. The endpoint key includes matrix and side IDs, because an `id` can recur in different fractions.
- Missing measured intermediates do **not** remove a possible path. Separate flags show mass-detection support in the broad Step 4/01 set, the Step 3/04 candidate set, and the Step 3 explicitly assigned M0,[M+H]+ subset. These are mass-level evidence, not proof of a chemical conversion.
- The Step 5b `inferred_formula` is shown for comparison but is not a path filter: it stores a selected annotation and can differ among parents.

Outputs are written to `results/step5d_all_reaction_paths/`.

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
from bisect import bisect_left, bisect_right
import ast
import csv
import json


def locate_v9_b():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents, here / 'Shuzhao_result' / 'v9_b'):
        if (candidate / 'results' / 'step4_predicted' / 'dhea_3step_products.json').is_file():
            return candidate
    raise FileNotFoundError('Could not locate Shuzhao_result/v9_b')


BASE = locate_v9_b()
RESULTS = BASE / 'results'
OUT = RESULTS / 'step5d_all_reaction_paths'
PARENTS = ('dhea', 'pregnenolone', 'progesterone', 'estrone', 'estradiol')
PARENT_MATRIX = {'dhea': 'DnHz', 'pregnenolone': 'DnHz', 'progesterone': 'DnHz',
                 'estrone': 'DnCl', 'estradiol': 'DnCl'}
MZ_PPM = 10.0
EPS = MZ_PPM / 1e6
print('v9_b directory:', BASE)

In [ ]:
def ion_text(value):
    if value is None:
        return ''
    if isinstance(value, list):
        return value[0] if len(value) == 1 else ';'.join(map(str, value))
    text = str(value).strip()
    if text.startswith('['):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return ion_text(parsed)
        except (ValueError, SyntaxError):
            pass
    return text


def split_ids(value):
    text = str(value or '').strip()
    return [] if text in ('', 'NA', 'nan', 'None') else [item.strip() for item in text.split(';') if item.strip()]


STEP3_PATHS = [RESULTS / 'step3_khipus' / f'{matrix}_{fraction}_features.json'
               for matrix in ('DnHz', 'DnCl') for fraction in ('pellets', 'supernatant')]
step3_features = {}
for path in STEP3_PATHS:
    for feature in json.loads(path.read_text()):
        key = (feature['source'], feature['id'])
        assert key not in step3_features, f'Duplicate source-qualified feature: {key}'
        step3_features[key] = feature

endpoints = {}
pairs_by_parent = defaultdict(list)
selected_rows_by_matrix = Counter()
for matrix in ('DnHz', 'DnCl'):
    path = RESULTS / 'step5b_combined_net_anno' / f'rxn_annotated_{matrix}_features.tsv'
    with path.open(newline='') as handle:
        for row_number, row in enumerate(csv.DictReader(handle, delimiter='\t'), start=1):
            if ion_text(row['ion_relation']) != 'M0,M+H+':
                continue
            selected_rows_by_matrix[matrix] += 1
            pellet_ids = split_ids(row['feature_id_pellets'])
            super_ids = split_ids(row['feature_id_supernatant'])
            endpoint_key = f"{matrix}|P:{';'.join(pellet_ids) or 'NA'}|S:{';'.join(super_ids) or 'NA'}"
            assert endpoint_key not in endpoints, f'Duplicate endpoint key: {endpoint_key}'
            sides = []
            for fraction, ids in (('pellets', pellet_ids), ('supernatant', super_ids)):
                source = f'{matrix}_{fraction}'
                for feature_id in ids:
                    feature = step3_features[(source, feature_id)]
                    sides.append({'source': source, 'id': feature_id, 'mz': float(feature['mz']),
                                  'rtime': float(feature['rtime']),
                                  'ion_relation_step3': ion_text(feature.get('ion_relation')),
                                  'parents': sorted(set(str(p).lower() for p in feature['substrate']))})
            assert sides, f'Endpoint has no original side: {endpoint_key}'
            endpoint = {'endpoint_key': endpoint_key, 'matrix': matrix, 'row_number': row_number,
                        'display_id': row['id'], 'mz_5c': float(row['mz']),
                        'inferred_formula_5c': row['inferred_formula'],
                        'label_origin_5c': ('step5b_list' if str(row['ion_relation']).startswith('[')
                                            else 'step3_literal'),
                        'feature_id_pellets': ';'.join(pellet_ids) or 'NA',
                        'feature_id_supernatant': ';'.join(super_ids) or 'NA',
                        'sides': sides}
            endpoints[endpoint_key] = endpoint
            for parent in sorted({p for side in sides for p in side['parents']}):
                assert PARENT_MATRIX[parent] == matrix
                parent_sides = [side for side in sides if parent in side['parents']]
                eligible_sides = [side for side in parent_sides
                                  if side['ion_relation_step3'] in ('', 'M0,M+H+')]
                pairs_by_parent[parent].append({**endpoint, 'parent': parent,
                                                'parent_sides': parent_sides,
                                                'eligible_sides': eligible_sides,
                                                'eligible': bool(eligible_sides),
                                                'qc_status': ('ok' if eligible_sides else 'isotope_or_other_ion_only')})

assert len(endpoints) == sum(selected_rows_by_matrix.values())
assert len({pair['endpoint_key'] for pairs in pairs_by_parent.values() for pair in pairs}) == len(endpoints)
print('Final M0,[M+H]+ endpoint rows:', dict(selected_rows_by_matrix), 'total', len(endpoints))
print('Parent-feature pairs:', {parent: len(pairs_by_parent[parent]) for parent in PARENTS},
      'total', sum(map(len, pairs_by_parent.values())))
print('QC-excluded parent pairs:', [(p['parent'], p['endpoint_key'])
                                   for pairs in pairs_by_parent.values() for p in pairs if not p['eligible']])

In [ ]:
class JsonTokenStream:
    # Incremental decoder: one Step 5a product at a time, avoiding a >1 GB in-memory JSON load.
    def __init__(self, handle, chunk_size=1 << 20):
        self.handle = handle
        self.chunk_size = chunk_size
        self.decoder = json.JSONDecoder()
        self.buffer = ''
        self.pos = 0
        self.eof = False

    def fill(self):
        self.buffer = self.buffer[self.pos:]
        self.pos = 0
        chunk = self.handle.read(self.chunk_size)
        if chunk:
            self.buffer += chunk
        else:
            self.eof = True

    def peek(self):
        while True:
            while self.pos < len(self.buffer) and self.buffer[self.pos].isspace():
                self.pos += 1
            if self.pos < len(self.buffer):
                return self.buffer[self.pos]
            if self.eof:
                raise EOFError('Unexpected end of Step 5a JSON')
            self.fill()

    def expect(self, character):
        found = self.peek()
        if found != character:
            raise ValueError(f'Expected {character!r}; found {found!r}')
        self.pos += 1

    def decode(self):
        while True:
            self.peek()
            try:
                value, end = self.decoder.raw_decode(self.buffer, self.pos)
                self.pos = end
                return value
            except json.JSONDecodeError:
                if self.eof:
                    raise
                self.fill()


def iter_step5a_products(path):
    with path.open(encoding='utf-8') as handle:
        stream = JsonTokenStream(handle)
        stream.expect('{')
        while stream.peek() != '}':
            stage = int(stream.decode())
            stream.expect(':')
            stream.expect('[')
            while stream.peek() != ']':
                yield stage, stream.decode()
                if stream.peek() == ',':
                    stream.expect(',')
                elif stream.peek() != ']':
                    raise ValueError(f'Unexpected array token in {path}')
            stream.expect(']')
            if stream.peek() == ',':
                stream.expect(',')
            elif stream.peek() != '}':
                raise ValueError(f'Unexpected object token in {path}')
        stream.expect('}')


def delta_key(reaction):
    return tuple(sorted((str(element), int(number)) for element, number in reaction['formula_dict'].items()
                        if int(number) != 0))


def delta_label(key):
    order = {'C': 0, 'H': 1, 'N': 2, 'O': 3, 'P': 4, 'S': 5}
    def term(element, count):
        return element + (str(count) if count != 1 else '')
    positive = ''.join(term(el, count) for el, count in sorted(key, key=lambda x: (order.get(x[0], 10), x[0]))
                       if count > 0)
    negative = ''.join(term(el, -count) for el, count in sorted(key, key=lambda x: (order.get(x[0], 10), x[0]))
                       if count < 0)
    return ('+' + positive if positive else '') + ('-' + negative if negative else '')


def reaction_labels(value):
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    text = str(value or '').strip()
    if text.startswith('['):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return reaction_labels(parsed)
        except (ValueError, SyntaxError):
            pass
    return [text] if text else []


def ancestry(product):
    nodes = []
    node = product
    while node.get('input_cpd') is not None:
        nodes.append(node)
        node = node['input_cpd']
    nodes.reverse()
    return node, nodes


def nearby(sorted_mzs, entries, predicted_mz):
    lo = bisect_left(sorted_mzs, predicted_mz / (1 + EPS))
    hi = bisect_right(sorted_mzs, predicted_mz / (1 - EPS))
    return [(entry, abs(predicted_mz - entry['mz']) / entry['mz'] * 1e6)
            for entry in entries[lo:hi]
            if abs(predicted_mz - entry['mz']) < entry['mz'] * EPS]


# Step 3 candidates provide two narrower intermediate-evidence levels than Step 5a/01.
intermediate_queries = defaultdict(list)
for (source, feature_id), feature in step3_features.items():
    ion = ion_text(feature.get('ion_relation'))
    if ion not in ('', 'M0,M+H+'):
        continue
    for parent in set(str(p).lower() for p in feature['substrate']):
        intermediate_queries[parent].append({'mz': float(feature['mz']), 'source': source,
                                             'id': feature_id, 'ion_relation': ion})
for parent in PARENTS:
    intermediate_queries[parent].sort(key=lambda x: x['mz'])


def intermediate_matches(parent, mz, excluded_side_keys):
    pool = intermediate_queries[parent]
    hits = nearby([q['mz'] for q in pool], pool, mz)
    eligible = [q for q, _ in hits if f"{q['source']}:{q['id']}" not in excluded_side_keys]
    confirmed_m0 = [q for q in eligible if q['ion_relation'] == 'M0,M+H+']
    return eligible, confirmed_m0

In [ ]:
all_routes = []
all_rule_annotations = defaultdict(lambda: {'forms': set(), 'labels': set(), 'sources': set()})
scan_summary = []

for parent in PARENTS:
    pairs = {pair['endpoint_key']: pair for pair in pairs_by_parent[parent] if pair['eligible']}
    side_queries = []
    for pair in pairs.values():
        for side in pair['eligible_sides']:
            side_queries.append({'mz': side['mz'], 'source': side['source'], 'id': side['id'],
                                 'endpoint_key': pair['endpoint_key']})
    side_queries.sort(key=lambda x: x['mz'])
    query_mzs = [entry['mz'] for entry in side_queries]
    predicted_path = RESULTS / 'step4_predicted' / f'{parent}_3step_products.json'
    retained_product_ids = set()
    routes = {}
    stage_counts = Counter()
    matched_product_count = 0
    cyclic_matching_products_excluded = 0
    cyclic_parent_endpoint_hits_excluded = 0

    for stage, product in iter_step5a_products(predicted_path):
        assert stage in (1, 2, 3)
        stage_counts[stage] += 1
        retained_product_ids.add(product['id'])
        hits = nearby(query_mzs, side_queries, float(product['mz']))
        if not hits:
            continue
        matched_product_count += 1
        root, nodes = ancestry(product)
        assert len(nodes) == stage, (parent, product['id'], stage)
        assert str(root['id']).lower() == parent, (parent, product['id'], root['id'])
        formula_chain = [root['formula']] + [node['formula'] for node in nodes]
        if len(set(formula_chain)) != len(formula_chain):
            cyclic_matching_products_excluded += 1
            cyclic_parent_endpoint_hits_excluded += len({entry['endpoint_key'] for entry, _ in hits})
            continue
        deltas = tuple(delta_key(node['reaction']) for node in nodes)
        assert all(deltas)
        per_endpoint_hits = defaultdict(list)
        for entry, ppm in hits:
            per_endpoint_hits[entry['endpoint_key']].append((entry, ppm))
        for endpoint_key, endpoint_hits in per_endpoint_hits.items():
            pair = pairs[endpoint_key]
            key = (endpoint_key, deltas)
            best_entry, best_ppm = min(endpoint_hits, key=lambda item: item[1])
            score = int(product['detection_score'])
            if key not in routes:
                routes[key] = {'parent': parent, 'pair': pair, 'deltas': deltas, 'steps': stage,
                               'root_formula': root['formula'],
                               'intermediate_formulas': [n['formula'] for n in nodes[:-1]],
                               'intermediate_mzs': [float(n['mz']) for n in nodes[:-1]],
                               'predicted_final_formula': product['formula'],
                               'best_ppm': best_ppm, 'example_product_id': product['id'],
                               'best_support_score': score,
                               'support_intermediate_ids': [n['id'] for n in nodes[:-1]],
                               'matched_side_keys': set(), 'raw_product_hits': 0,
                               'final_formula_variants': set()}
            route = routes[key]
            assert route['steps'] == stage
            route['raw_product_hits'] += 1
            route['matched_side_keys'].update(f"{e['source']}:{e['id']}" for e, _ in endpoint_hits)
            route['final_formula_variants'].add(product['formula'])
            if best_ppm < route['best_ppm']:
                route['best_ppm'] = best_ppm
                route['example_product_id'] = product['id']
                route['intermediate_mzs'] = [float(n['mz']) for n in nodes[:-1]]
            if score > route['best_support_score']:
                route['best_support_score'] = score
                route['support_intermediate_ids'] = [n['id'] for n in nodes[:-1]]
            for node, delta in zip(nodes, deltas):
                annotation = all_rule_annotations[delta]
                annotation['forms'].add(str(node['reaction']['formula_difference']))
                annotation['labels'].update(reaction_labels(node['reaction'].get('reactionIDs')))
                annotation['sources'].add(str(node['reaction'].get('source', '')))

    for route in routes.values():
        pair = route['pair']
        support_01 = [product_id in retained_product_ids for product_id in route['support_intermediate_ids']]
        assert sum(support_01) + 1 == route['best_support_score']
        excluded = {f"{side['source']}:{side['id']}" for side in pair['parent_sides']}
        support_04, support_step3_m0 = [], []
        support_04_ids, support_step3_m0_ids = [], []
        for mz in route['intermediate_mzs']:
            candidates, primary_m0 = intermediate_matches(parent, mz, excluded)
            support_04.append(bool(candidates))
            support_step3_m0.append(bool(primary_m0))
            support_04_ids.append([f"{item['source']}:{item['id']}" for item in candidates])
            support_step3_m0_ids.append([f"{item['source']}:{item['id']}" for item in primary_m0])
        route['intermediate_01_flags'] = support_01
        route['intermediate_04_flags'] = support_04
        route['intermediate_step3_m0h_flags'] = support_step3_m0
        route['intermediate_04_feature_ids'] = support_04_ids
        route['intermediate_step3_m0h_feature_ids'] = support_step3_m0_ids
        assert len(route['final_formula_variants']) == 1, (parent, pair['endpoint_key'], route['deltas'])
        all_routes.append(route)

    scan_summary.append({'parent': parent, 'predicted_step1': stage_counts[1],
                         'predicted_step2': stage_counts[2], 'predicted_step3': stage_counts[3],
                         'predicted_products_matching_endpoints': matched_product_count,
                         'cyclic_matching_products_excluded': cyclic_matching_products_excluded,
                         'cyclic_parent_endpoint_hits_excluded': cyclic_parent_endpoint_hits_excluded,
                         'unique_parent_endpoint_paths': len(routes)})
    print(parent, 'predicted', dict(stage_counts), 'matched products', matched_product_count,
          'cyclic products excluded', cyclic_matching_products_excluded,
          'unique endpoint paths', len(routes))
    del retained_product_ids, routes

print('Total unique parent-endpoint paths:', len(all_routes))

In [ ]:
def json_text(value):
    return json.dumps(value, ensure_ascii=False, separators=(',', ':'))


def write_tsv(path, fieldnames, rows):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, delimiter='\t', extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


OUT.mkdir(parents=True, exist_ok=True)
route_rows = []
edge_data = defaultdict(lambda: {'endpoint_keys': set(), 'path_occurrences': 0})
route_count_by_pair = Counter()
route_formula_agree_by_pair = Counter()
route_step_count_by_pair = defaultdict(Counter)
route_01_support_by_pair = Counter()
route_04_support_by_pair = Counter()
route_m0h_support_by_pair = Counter()
unique_delta_sequences_by_parent = defaultdict(set)

for route in sorted(all_routes, key=lambda r: (r['parent'], r['pair']['endpoint_key'], r['steps'], r['deltas'])):
    pair = route['pair']
    parent, endpoint_key, steps = route['parent'], pair['endpoint_key'], route['steps']
    changes = [delta_label(delta) for delta in route['deltas']]
    formulas = [route['root_formula']] + route['intermediate_formulas'] + [route['predicted_final_formula']]
    assert len(formulas) == steps + 1
    route_count_by_pair[(parent, endpoint_key)] += 1
    if route['predicted_final_formula'] == pair['inferred_formula_5c']:
        route_formula_agree_by_pair[(parent, endpoint_key)] += 1
    route_step_count_by_pair[(parent, endpoint_key)][steps] += 1
    unique_delta_sequences_by_parent[parent].add(route['deltas'])
    if steps > 1 and all(route['intermediate_01_flags']):
        route_01_support_by_pair[(parent, endpoint_key)] += 1
    if steps > 1 and all(route['intermediate_04_flags']):
        route_04_support_by_pair[(parent, endpoint_key)] += 1
    if steps > 1 and all(route['intermediate_step3_m0h_flags']):
        route_m0h_support_by_pair[(parent, endpoint_key)] += 1
    for source_formula, target_formula, change in zip(formulas[:-1], formulas[1:], changes):
        edge = edge_data[(parent, source_formula, target_formula, change)]
        edge['endpoint_keys'].add(endpoint_key)
        edge['path_occurrences'] += 1
    route_rows.append({
        'parent': parent, 'endpoint_key': endpoint_key, 'feature_id_5c': pair['display_id'],
        'feature_id_pellets': pair['feature_id_pellets'],
        'feature_id_supernatant': pair['feature_id_supernatant'], 'matrix': pair['matrix'],
        'ion_label_origin_5c': pair['label_origin_5c'],
        'any_step3_confirmed_m0h_side': any(s['ion_relation_step3'] == 'M0,M+H+'
                                               for s in pair['parent_sides']),
        'endpoint_mz_5c': pair['mz_5c'], 'matched_endpoint_sides': ';'.join(sorted(route['matched_side_keys'])),
        'steps': steps, 'reaction_changes': json_text(changes), 'formula_sequence': json_text(formulas),
        'intermediate_formulas': json_text(route['intermediate_formulas']),
        'predicted_final_formula': route['predicted_final_formula'],
        'inferred_formula_5c': pair['inferred_formula_5c'],
        'matches_5c_inferred_formula': route['predicted_final_formula'] == pair['inferred_formula_5c'],
        'best_abs_ppm': round(route['best_ppm'], 6), 'raw_step5a_product_hits': route['raw_product_hits'],
        'example_step5a_product_id': route['example_product_id'],
        'step5a_detection_score_max': route['best_support_score'],
        'intermediate_01_detected': json_text(route['intermediate_01_flags']),
        'all_intermediates_01_detected': all(route['intermediate_01_flags']),
        'intermediate_04_candidate_detected': json_text(route['intermediate_04_flags']),
        'all_intermediates_04_candidate_detected': all(route['intermediate_04_flags']),
        'intermediate_04_candidate_feature_ids': json_text(route['intermediate_04_feature_ids']),
        'intermediate_step3_m0h_detected': json_text(route['intermediate_step3_m0h_flags']),
        'all_intermediates_step3_m0h_detected': all(route['intermediate_step3_m0h_flags']),
        'intermediate_step3_m0h_feature_ids': json_text(route['intermediate_step3_m0h_feature_ids']),
    })

feature_rows = []
for parent in PARENTS:
    for pair in sorted(pairs_by_parent[parent], key=lambda p: p['endpoint_key']):
        key = (parent, pair['endpoint_key'])
        feature_rows.append({
            'parent': parent, 'endpoint_key': pair['endpoint_key'], 'feature_id_5c': pair['display_id'],
            'feature_id_pellets': pair['feature_id_pellets'],
            'feature_id_supernatant': pair['feature_id_supernatant'], 'matrix': pair['matrix'],
            'endpoint_mz_5c': pair['mz_5c'], 'inferred_formula_5c': pair['inferred_formula_5c'],
            'ion_label_origin_5c': pair['label_origin_5c'],
            'any_step3_confirmed_m0h_side': any(s['ion_relation_step3'] == 'M0,M+H+'
                                                   for s in pair['parent_sides']),
            'parent_side_ids': ';'.join(f"{s['source']}:{s['id']}" for s in pair['parent_sides']),
            'parent_side_step3_ion_relations': json_text([s['ion_relation_step3'] for s in pair['parent_sides']]),
            'eligible_for_primary_count': pair['eligible'], 'qc_status': pair['qc_status'],
            'one_step_paths': route_step_count_by_pair[key][1],
            'two_step_paths': route_step_count_by_pair[key][2],
            'three_step_paths': route_step_count_by_pair[key][3],
            'all_candidate_paths': route_count_by_pair[key],
            'paths_matching_5c_inferred_formula': route_formula_agree_by_pair[key],
            'multi_step_paths_all_intermediates_01_detected': route_01_support_by_pair[key],
            'multi_step_paths_all_intermediates_04_candidate_detected': route_04_support_by_pair[key],
            'multi_step_paths_all_intermediates_step3_m0h_detected': route_m0h_support_by_pair[key],
        })

parent_rows = []
for parent in PARENTS:
    features = [row for row in feature_rows if row['parent'] == parent]
    paths = [row for row in route_rows if row['parent'] == parent]
    parent_rows.append({
        'parent': parent, 'endpoint_pairs_total': len(features),
        'endpoint_pairs_eligible': sum(row['eligible_for_primary_count'] for row in features),
        'endpoint_pairs_with_paths': sum(row['all_candidate_paths'] > 0 for row in features),
        'one_step_paths': sum(row['one_step_paths'] for row in features),
        'two_step_paths': sum(row['two_step_paths'] for row in features),
        'three_step_paths': sum(row['three_step_paths'] for row in features),
        'all_parent_endpoint_paths': len(paths),
        'paths_matching_5c_inferred_formula': sum(
            row['paths_matching_5c_inferred_formula'] for row in features),
        'distinct_formula_change_sequences': len(unique_delta_sequences_by_parent[parent]),
        'distinct_formula_edges': sum(edge_parent == parent for edge_parent, *_ in edge_data),
        'multi_step_paths_all_intermediates_01_detected': sum(
            row['multi_step_paths_all_intermediates_01_detected'] for row in features),
        'multi_step_paths_all_intermediates_04_candidate_detected': sum(
            row['multi_step_paths_all_intermediates_04_candidate_detected'] for row in features),
        'multi_step_paths_all_intermediates_step3_m0h_detected': sum(
            row['multi_step_paths_all_intermediates_step3_m0h_detected'] for row in features),
    })

edge_rows = [{'parent': parent, 'source_formula': source, 'target_formula': target,
              'formula_difference': change, 'endpoint_feature_count': len(data['endpoint_keys']),
              'path_occurrences': data['path_occurrences']}
             for (parent, source, target, change), data in sorted(edge_data.items())]
rule_rows = [{'formula_difference_normalized': delta_label(delta),
              'source_spellings': json_text(sorted(annotation['forms'])),
              'source_tables': json_text(sorted(annotation['sources'])),
              'candidate_reaction_name_count': len(annotation['labels']),
              'candidate_reaction_names': json_text(sorted(annotation['labels']))}
             for delta, annotation in sorted(all_rule_annotations.items())]

write_tsv(OUT / 'all_reaction_paths.tsv', list(route_rows[0]) if route_rows else [], route_rows)
write_tsv(OUT / 'feature_path_summary.tsv', list(feature_rows[0]), feature_rows)
write_tsv(OUT / 'parent_path_summary.tsv', list(parent_rows[0]), parent_rows)
write_tsv(OUT / 'unique_formula_edges.tsv', list(edge_rows[0]) if edge_rows else [], edge_rows)
write_tsv(OUT / 'reaction_rule_lookup.tsv', list(rule_rows[0]) if rule_rows else [], rule_rows)
write_tsv(OUT / 'scan_summary.tsv', list(scan_summary[0]), scan_summary)

assert len(feature_rows) == sum(len(pairs_by_parent[parent]) for parent in PARENTS)
assert sum(row['all_candidate_paths'] for row in feature_rows) == len(route_rows)
assert all(row['one_step_paths'] + row['two_step_paths'] + row['three_step_paths'] == row['all_candidate_paths']
           for row in feature_rows)
assert all(row['all_parent_endpoint_paths'] == row['one_step_paths'] + row['two_step_paths'] + row['three_step_paths']
           for row in parent_rows)
print('Output directory:', OUT)
for row in parent_rows:
    print(row['parent'], 'eligible endpoints', row['endpoint_pairs_eligible'],
          '1/2/3-step paths', row['one_step_paths'], row['two_step_paths'], row['three_step_paths'],
          'unique edges', row['distinct_formula_edges'],
          'paths agreeing with 5c formula', row['paths_matching_5c_inferred_formula'])
print('QC: endpoint rows', len(endpoints), 'parent-feature pairs', len(feature_rows),
      'excluded isotope-only pairs', sum(not row['eligible_for_primary_count'] for row in feature_rows),
      'path rows', len(route_rows))
print('Examples with both direct and two-step paths:')
for row in [r for r in feature_rows if r['one_step_paths'] and r['two_step_paths']][:5]:
    print(row['parent'], row['endpoint_key'], '1-step', row['one_step_paths'], '2-step', row['two_step_paths'])

## Reaction-difference frequency among final annotated features

Count **distinct `(steroid parent, merged endpoint, normalized formula difference)` triples**. A feature contributes once to a reaction difference even if that difference appears in several of its candidate paths or twice within one path. The global count uses parent-feature pairs, so one merged feature present in two steroid groups contributes once to each parent.

Two rankings are exported:

1. **All candidate paths:** includes routes with unobserved intermediates. No observed/unobserved split is applied to these frequencies.
2. **Intermediate-supported paths:** only paths with every intermediate mass detected in the parent-specific broad `01` feature tables, as recorded by Step 4 (`all_intermediates_01_detected`). One-step paths have no intermediate and remain eligible. This is mass-level detection support, not a confirmed M0 or chemical reaction.

Both files include the full ranked list overall and within each steroid parent. Rows ranked 1–10 or 1–15 can be used directly for Top 10/15 plots.

In [ ]:
from collections import Counter

all_associations = set()
supported_associations = set()
with (OUT / 'all_reaction_paths.tsv').open(newline='', encoding='utf-8') as handle:
    for row in csv.DictReader(handle, delimiter='\t'):
        deltas = set(json.loads(row['reaction_changes']))
        assert deltas and all(delta.startswith(('+', '-')) for delta in deltas)
        for delta in deltas:
            association = (row['parent'], row['endpoint_key'], delta)
            all_associations.add(association)
            if row['all_intermediates_01_detected'] == 'True':
                supported_associations.add(association)

assert supported_associations <= all_associations
eligible_pairs = {(row['parent'], row['endpoint_key']) for row in feature_rows
                  if row['eligible_for_primary_count']}
assert all((parent, endpoint_key) in eligible_pairs
           for parent, endpoint_key, _ in all_associations)


def frequency_rows(associations):
    rows = []
    for scope in ('ALL', *PARENTS):
        selected = (associations if scope == 'ALL'
                    else {(parent, endpoint_key, delta)
                          for parent, endpoint_key, delta in associations if parent == scope})
        counts = Counter(delta for _, _, delta in selected)
        denominator = (len(eligible_pairs) if scope == 'ALL'
                       else sum(parent == scope for parent, _ in eligible_pairs))
        for rank, (delta, count) in enumerate(sorted(counts.items(), key=lambda item: (-item[1], item[0])), start=1):
            rows.append({'scope': scope, 'rank': rank, 'formula_difference': delta,
                         'parent_feature_count': count,
                         'eligible_parent_feature_pairs_in_scope': denominator,
                         'percent_of_eligible_pairs': round(100 * count / denominator, 2)})
    return rows


all_frequency = frequency_rows(all_associations)
supported_frequency = frequency_rows(supported_associations)
write_tsv(OUT / 'reaction_difference_frequency_all_paths.tsv', list(all_frequency[0]), all_frequency)
write_tsv(OUT / 'reaction_difference_frequency_supported_paths.tsv',
          list(supported_frequency[0]), supported_frequency)

all_by_scope_delta = {(row['scope'], row['formula_difference']): row['parent_feature_count']
                      for row in all_frequency}
assert all(row['parent_feature_count'] <= all_by_scope_delta[(row['scope'], row['formula_difference'])]
           for row in supported_frequency)
print('Distinct parent-feature–reaction-difference associations:',
      'all paths', len(all_associations), 'intermediate-supported paths', len(supported_associations))
for title, rows in [('All candidate paths', all_frequency),
                    ('Intermediate-supported paths', supported_frequency)]:
    print('\n' + title + ' — overall Top 15')
    for row in rows:
        if row['scope'] == 'ALL' and row['rank'] <= 15:
            print(f"{row['rank']:2d}. {row['formula_difference']:>12s}  {row['parent_feature_count']:4d} parent-feature pairs")


## Separate rankings for each path length

The preceding two frequency tables combine **all path lengths from 1 through 3**. The following three files restrict the path length to exactly one, two, or three steps. Each file has `mode=all` and `mode=intermediate_supported` rankings, overall and per parent. A one-step route has no intermediate, so its two modes have identical counts. The same parent–feature–reaction-difference association may appear at multiple path lengths and is counted once within each length-specific table.

In [ ]:
step_associations = {steps: {'all': set(), 'intermediate_supported': set()}
                     for steps in (1, 2, 3)}
step_path_rows = Counter()
supported_step_path_rows = Counter()
with (OUT / 'all_reaction_paths.tsv').open(newline='', encoding='utf-8') as handle:
    for row in csv.DictReader(handle, delimiter='\t'):
        steps = int(row['steps'])
        if steps not in step_associations:
            continue
        step_path_rows[steps] += 1
        supported = row['all_intermediates_01_detected'] == 'True'
        if supported:
            supported_step_path_rows[steps] += 1
        for delta in set(json.loads(row['reaction_changes'])):
            association = (row['parent'], row['endpoint_key'], delta)
            step_associations[steps]['all'].add(association)
            if supported:
                step_associations[steps]['intermediate_supported'].add(association)

assert step_path_rows[1] == supported_step_path_rows[1]
assert step_associations[1]['all'] == step_associations[1]['intermediate_supported']
for steps in (1, 2, 3):
    assert step_associations[steps]['intermediate_supported'] <= step_associations[steps]['all']
    rows = []
    for mode in ('all', 'intermediate_supported'):
        rows.extend({'path_steps': steps, 'mode': mode, **record}
                    for record in frequency_rows(step_associations[steps][mode]))
    output = OUT / f'reaction_difference_frequency_{steps}step.tsv'
    write_tsv(output, list(rows[0]), rows)
    print(f'\n{steps}-step paths: {step_path_rows[steps]} all, '
          f'{supported_step_path_rows[steps]} intermediate-supported; output: {output.name}')
    for mode in ('all', 'intermediate_supported'):
        associations = step_associations[steps][mode]
        print(f"  {mode}: {len(associations)} distinct parent-feature–reaction-difference associations")
        top = [row for row in rows if row['scope'] == 'ALL' and row['mode'] == mode and row['rank'] <= 15]
        print('  overall Top 15:', [(row['formula_difference'], row['parent_feature_count']) for row in top])
